In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# --- LOCAL MODULES ---
import sys
sys.path.append('..') # Allow importing from src
from src.utils import get_device
from src.data import SemanticStackDataset
from src.models import get_resnet_model

# --- CONFIGURATION ---
DATA_DIR = "../data/mask_tensors"
TABULAR_PATH = "../data/processed/final_dataset.csv"
MODEL_SAVE_PATH = "../models/best_resnet.pth"
BATCH_SIZE = 256
LEARNING_RATE = 0.001
EPOCHS = 20

DEVICE = get_device()

# Check for XGBoost
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("⚠️ XGBoost library not found. Skipping baseline.")

In [ ]:
# --- BASELINE: XGBoost on Tabular Data ---
xgb_acc, xgb_auc = 0.5, 0.5 # Defaults

if os.path.exists(TABULAR_PATH) and XGB_AVAILABLE:
    print("🤖 Running Tabular Baseline (XGBoost)...")
    
    # Load
    df = pd.read_csv(TABULAR_PATH)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df = df.dropna()
    
    # Feature Selection (Exclude ID/Lat/Lon/Filename)
    feature_cols = [c for c in df.columns if c not in ['id', 'target', 'lat', 'lon', 'filename']]
    X_tab = df[feature_cols].values
    y_tab = df['target'].values
    
    # Split
    X_tab_train, X_tab_test, y_tab_train, y_tab_test = train_test_split(
        X_tab, y_tab, test_size=0.2, random_state=42, stratify=y_tab
    )
    
    # Train
    xgb = XGBClassifier(n_estimators=100, learning_rate=0.05, eval_metric='logloss')
    xgb.fit(X_tab_train, y_tab_train)
    
    # Score
    y_tab_pred = xgb.predict(X_tab_test)
    y_tab_prob = xgb.predict_proba(X_tab_test)[:, 1]
    
    xgb_acc = accuracy_score(y_tab_test, y_tab_pred)
    xgb_auc = roc_auc_score(y_tab_test, y_tab_prob)
    
    print("-" * 30)
    print(f"📊 TABULAR BASELINE (XGBoost):")
    print(f"Accuracy: {xgb_acc:.4f}")
    print(f"AUC:      {xgb_auc:.4f}")
    print("-" * 30)
else:
    print("⚠️ Tabular dataset not found or XGBoost missing. Skipping baseline.")

In [ ]:
# --- PREPARE DATA ---
if not os.path.exists(DATA_DIR):
    print(f"⚠️ Warning: Folder {DATA_DIR} not found.")
    all_files = []
else:
    all_files = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith('.npy')]
    print(f"Found {len(all_files)} tensor files.")

if len(all_files) > 0:
    train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)

    # Augmentations
    train_transforms = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(90),
    ])

    # Optimized Loaders
    # Note: num_workers=0 avoids multiprocessing issues in notebooks
    train_loader = DataLoader(
        SemanticStackDataset(train_files, transform=train_transforms), 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=0, 
        pin_memory=False
    )
    
    val_loader = DataLoader(
        SemanticStackDataset(val_files), 
        batch_size=BATCH_SIZE, 
        num_workers=0,
        pin_memory=False
    )
    print(f"Data Loaded: {len(train_files)} Train, {len(val_files)} Validation")

In [ ]:
model = get_resnet_model(DEVICE)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')

In [ ]:
if len(all_files) > 0:
    print(f"🚀 Starting Training for {EPOCHS} epochs on {DEVICE}...")
    
    for epoch in range(EPOCHS):
        # --- TRAIN ---
        model.train()
        running_loss = 0.0
        
        # Progress bar for Training
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
        
        for inputs, labels in train_loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            train_loop.set_postfix(loss=loss.item())
            
        # --- VALIDATE ---
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        # Progress bar for Validation
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
        
        with torch.no_grad():
            for inputs, labels in val_loop:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                predicted = (outputs > 0.5).float()
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        
        # Stats
        epoch_loss = running_loss / len(train_loader)
        epoch_val_loss = val_loss / len(val_loader)
        epoch_acc = correct / total
        
        history['train_loss'].append(epoch_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_acc)
        
        print(f"✅ Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_acc:.4f}")
        
        # Save Best
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"   --> 💾 Best model saved! (New Low: {best_val_loss:.4f})")

    # Plot History
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Val')
    plt.title('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['val_acc'], label='Validation Accuracy', color='green')
    plt.title('Accuracy')
    plt.legend()
    plt.show()

else:
    print("❌ No data found.")

In [ ]:
if len(all_files) > 0:
    print("\n🔍 Running Final Evaluation...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()

    y_true = []
    y_pred_probs = []

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Final Eval"):
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred_probs.extend(outputs.cpu().numpy())

    y_true = np.array(y_true).flatten()
    y_pred_probs = np.array(y_pred_probs).flatten()
    y_pred_cls = (y_pred_probs > 0.5).astype(int)

    cnn_acc = accuracy_score(y_true, y_pred_cls)
    cnn_auc = roc_auc_score(y_true, y_pred_probs)

    print(f"\n🏆 CNN (ResNet18) RESULTS:")
    print(f"Accuracy: {cnn_acc:.4f}")
    print(f"AUC Score: {cnn_auc:.4f}")
    print("\n" + classification_report(y_true, y_pred_cls))

    # --- VISUAL COMPARISON ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred_cls)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Survived', 'Burned'], 
                yticklabels=['Survived', 'Burned'], ax=axes[0])
    axes[0].set_title("CNN Confusion Matrix")

    # Bar Chart
    models_list = ['XGBoost (Tabular)', 'ResNet (Visual)']
    accuracies = [xgb_acc, cnn_acc]
    aucs = [xgb_auc, cnn_auc]
    x = np.arange(len(models_list))
    width = 0.35

    rects1 = axes[1].bar(x - width/2, accuracies, width, label='Accuracy', color='skyblue')
    rects2 = axes[1].bar(x + width/2, aucs, width, label='AUC', color='orange')

    axes[1].set_ylabel('Score')